Purpose: To develop and validate the split script.

- Import create_split.py
- Run with real files
- Check counts
- Check event balance
- Confirm no ID overlap

In [14]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import argparse

In [15]:
def load_inputs(sample_ids_path, survival_path, event_col, id_col="sample"):
    ids = pd.read_csv(sample_ids_path)[id_col].astype(str)
    surv = pd.read_csv(survival_path, sep='\t')[[id_col, event_col]].astype({id_col: str})
    id_outcome_df = pd.DataFrame({id_col: ids}).merge(surv, on=id_col, how="inner")
    return id_outcome_df

id_outcome_df = load_inputs("../data/interim/sample_ids_filtered.csv", "../data/raw/TCGA-BRCA.survival.tsv.gz", event_col="OS")
display(id_outcome_df.shape, id_outcome_df.head())

(290, 2)

,sample,OS
0,TCGA-3C-AAAU-01A,0
1,TCGA-3C-AALI-01A,0
2,TCGA-A1-A0SK-01A,1
3,TCGA-A2-A04N-01A,0
4,TCGA-A2-A04Q-01A,0


In [16]:
def make_splits(df, event_col, val_size=0.15, test_size=0.15, seed=42, id_col="sample"):
    y = df[event_col].astype(int)

    train_ids, temp_ids = train_test_split(
        df[id_col],
        test_size=val_size + test_size,
        stratify=y,
        random_state=seed,
    )

    temp = df.set_index(id_col).loc[temp_ids]
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=test_size / (val_size + test_size),
        stratify=temp[event_col].astype(int),
        random_state=seed,
    )

    return train_ids, val_ids, test_ids

train_ids, val_ids, test_ids = make_splits(id_outcome_df, event_col="OS", val_size=0.15, test_size=0.15, seed=42)

In [17]:
def validate_and_summarize_splits(df, train_ids, val_ids, test_ids, event_col, id_col="sample", val_size=0.15, test_size=0.15, size_tol=1):
    # 1) No duplicate sample IDs in source frame
    assert not df[id_col].duplicated().any(), "Duplicate sample IDs found in input dataframe"

    # 2) No missing values in key columns
    key_cols = [id_col, event_col]
    missing_counts = df[key_cols].isna().sum()
    assert (missing_counts == 0).all(), f"Missing values detected in key columns: {missing_counts.to_dict()}"

    # 3) No overlap between train/val/test split IDs
    train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)
    assert train_set.isdisjoint(val_set), "Overlap found between train and val IDs"
    assert train_set.isdisjoint(test_set), "Overlap found between train and test IDs"
    assert val_set.isdisjoint(test_set), "Overlap found between val and test IDs"

    # 4) Dataset sizes match expectations
    n_total = len(df)
    expected_test = round(n_total * test_size)
    expected_val = round(n_total * val_size)
    expected_train = n_total - expected_val - expected_test

    actual_train, actual_val, actual_test = len(train_ids), len(val_ids), len(test_ids)
    assert actual_train + actual_val + actual_test == n_total, "Split sizes do not sum to total dataset size"
    assert abs(actual_train - expected_train) <= size_tol, f"Train size mismatch: expected ~{expected_train}, got {actual_train}"
    assert abs(actual_val - expected_val) <= size_tol, f"Val size mismatch: expected ~{expected_val}, got {actual_val}"
    assert abs(actual_test - expected_test) <= size_tol, f"Test size mismatch: expected ~{expected_test}, got {actual_test}"

    y = df.set_index(id_col)[event_col].astype(int)
    print("All validation checks passed.")
    print(f"n_total={n_total}, train={actual_train}, val={actual_val}, test={actual_test}")
    print("event rates:", y.loc[train_ids].mean(), y.loc[val_ids].mean(), y.loc[test_ids].mean())

validate_and_summarize_splits(id_outcome_df, train_ids, val_ids, test_ids, event_col="OS", id_col="sample", val_size=0.15, test_size=0.15)

All validation checks passed.
n_total=290, train=203, val=43, test=44
event rates: 0.3251231527093596 0.32558139534883723 0.3181818181818182


In [18]:
def save_splits(train_ids, val_ids, test_ids, outdir, id_col="sample"):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    pd.Series(train_ids, name=id_col).to_csv(outdir / "train_ids.csv", index=False)
    pd.Series(val_ids,   name=id_col).to_csv(outdir / "val_ids.csv",   index=False)
    pd.Series(test_ids,  name=id_col).to_csv(outdir / "test_ids.csv",  index=False)

save_splits(train_ids, val_ids, test_ids, outdir="../data/processed/splits/os_seed42_v15_t15")

In [20]:
# Test scripts/create_split.py main() via CLI
!python ../scripts/create_split.py \
    --sample-ids-path ../data/interim/sample_ids_filtered.csv \
    --survival-path ../data/raw/TCGA-BRCA.survival.tsv.gz \
    --event-col OS \
    --id-col sample \
    --val-size 0.15 \
    --test-size 0.15 \
    --seed 42 \
    --size-tol 1 \
    --outdir ../data/processed/splits/os_seed42_v15_t15_cli_test


All validation checks passed.
n_total=290, train=203, val=43, test=44
event rates: 0.3251231527093596 0.32558139534883723 0.3181818181818182
